# AI Hub 데이터셋 기반 STT 성능 벤치마크 — faster-whisper(small) vs SenseVoiceSmall

데이터셋: **141. 의료진 및 환자 음성** (datasetkey: `71411`)
- 라벨링데이터: `48745`
- 원천데이터: `48746` ~ `48758`

AI Hub는 해외/클라우드 IP에서의 다운로드를 차단해 Colab에서 `aihubshell`로 직접 받을 수 없습니다. 대신 로컬에 다운로드한 뒤 `rclone`으로 Google Drive(`colab/audiodata`)에 업로드해둘았고, 이 노트북은 Drive를 마운트해 바로 사용합니다.

한국어 의료 음성 데이터로 두 STT 모델의 실시간성과 정확도를 비교합니다.

**측정 지표**
- Latency (초): 오디오 1개당 추론 소요 시간
- RTF (Real-Time Factor) = 추론 시간 / 오디오 길이 — 1보다 작을수록 실시간보다 빠름
- CER (Character Error Rate): `jiwer` 기반 한국어 음절 오류율

**비교 대상**
- `faster-whisper` (`small`, GPU, float16)
- `SenseVoiceSmall` (FunASR/ModelScope, GPU)

실행 전에 Colab 상단 메뉴에서 **런타임 유형을 GPU로 변경**하세요 (런타임 > 런타임 유형 변경).

## 1. 환경 설정 및 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q faster-whisper funasr modelscope torchaudio jiwer librosa pandas tabulate

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 잡히지 않았습니다 — 상단 메뉴 [런타임 > 런타임 유형 변경]에서 GPU를 선택하세요.")

### 압축 해제

`colab/audiodata` 밑에 올라온 Validation 원천/라벨 zip을 I/O 속도 확보를 위해 Drive가 아닌 로컬(`/content/data`) 디스크에 푸는다.
경로가 다르면 `DRIVE_DATA_ROOT`만 수정하면 된다.

In [ ]:
import os

DRIVE_DATA_ROOT = "/content/drive/MyDrive/colab/audiodata/비대면 진료를 위한 의료진 및 환자 음성/Validation"
SOURCE_ZIP = f"{DRIVE_DATA_ROOT}/[V원천]의료진_간호사_1.zip"
LABEL_ZIP = f"{DRIVE_DATA_ROOT}/[V]라벨링데이터.zip"

EXTRACT_ROOT = "/content/data"
AUDIO_DIR = f"{EXTRACT_ROOT}/source"
LABEL_DIR = f"{EXTRACT_ROOT}/label"

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

!unzip -q -o -O UTF-8 "{SOURCE_ZIP}" -d "{AUDIO_DIR}"
!unzip -q -o -O UTF-8 "{LABEL_ZIP}" -d "{LABEL_DIR}"

print("Source files:", sum(len(f) for _, _, f in os.walk(AUDIO_DIR)))
print("Label files:", sum(len(f) for _, _, f in os.walk(LABEL_DIR)))

## 2. 모델 준비

In [ ]:
from faster_whisper import WhisperModel

fw_model = WhisperModel("small", device="cuda", compute_type="float16")
print("faster-whisper(small) loaded.")

In [ ]:
from funasr import AutoModel
from funasr.utils.postprocess_utils import rich_transcription_postprocess

sv_model = AutoModel(
    model="iic/SenseVoiceSmall",
    trust_remote_code=True,
    device="cuda:0",
)
print("SenseVoiceSmall loaded.")

## 3. 오디오-라벨 페어링

In [ ]:
from pathlib import Path

AUDIO_EXTS = (".wav", ".flac", ".pcm")
LABEL_EXTS = (".json", ".txt")

audio_files = [p for p in Path(AUDIO_DIR).rglob("*") if p.suffix.lower() in AUDIO_EXTS]
label_files = [p for p in Path(LABEL_DIR).rglob("*") if p.suffix.lower() in LABEL_EXTS]

label_by_stem = {p.stem: p for p in label_files}

pairs = []
unmatched = 0
for audio_path in audio_files:
    label_path = label_by_stem.get(audio_path.stem)
    if label_path is None:
        unmatched += 1
        continue
    pairs.append((audio_path, label_path))

print(f"오디오 파일: {len(audio_files)}개")
print(f"라벨 파일: {len(label_files)}개")
print(f"매칭된 쌍: {len(pairs)}개  (매칭 안 된 오디오: {unmatched}개)")

### 라벨 스키마 확인

아래 셀에서 라벨 파일 하나를 열어 실제 필드 구조를 확인하세요. 바로 아래 `load_transcript()`가 흔한 AI Hub 필드명을 자동으로 찾지만, 데이터셋마다 스키마가 달라 실패할 수 있습니다 — 그럴 경우 출력된 raw 내용을 보고 `_TEXT_KEY_CANDIDATES`에 실제 필드명을 추가하세요.

In [ ]:
import json

if pairs:
    sample_label_path = pairs[0][1]
    print("Sample label file:", sample_label_path)
    if sample_label_path.suffix.lower() == ".json":
        with open(sample_label_path, encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2)[:2000])
    else:
        print(sample_label_path.read_text(encoding="utf-8")[:2000])
else:
    print("매칭된 쌍이 없습니다 — 압축 해제 경로/파일명을 확인하세요.")

In [ ]:
import re

# AI Hub STT 라벨 JSON은 데이터셋마다 필드명이 다릅니다.
# 위 셀에서 출력된 실제 구조를 보고, 여기에 맞는 키를 추가하세요.
_TEXT_KEY_CANDIDATES = ["전사정보", "transcription", "text", "TransLabelText", "orgtext", "standard"]

def _find_text_value(obj, depth=0):
    if depth > 6:
        return None
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for key in _TEXT_KEY_CANDIDATES:
            if key in obj:
                found = _find_text_value(obj[key], depth + 1)
                if found:
                    return found
        for value in obj.values():
            found = _find_text_value(value, depth + 1)
            if found:
                return found
    if isinstance(obj, list):
        for item in obj:
            found = _find_text_value(item, depth + 1)
            if found:
                return found
    return None

def load_transcript(label_path: Path) -> str:
    if label_path.suffix.lower() == ".txt":
        return label_path.read_text(encoding="utf-8").strip()
    with open(label_path, encoding="utf-8") as f:
        data = json.load(f)
    text = _find_text_value(data)
    if not text:
        raise ValueError(f"'{label_path}'에서 전사 텍스트를 찾지 못했습니다 — _TEXT_KEY_CANDIDATES에 실제 필드명을 추가하세요.")
    return text.strip()

def normalize_text(text: str) -> str:
    # CER 비교 전 공백/문장부호 정규화 (필요에 따라 조정)
    text = re.sub(r"[^\uac00-\ud7a30-9a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 4. 벤치마크 실행

In [ ]:
import random

random.seed(42)
SAMPLE_SIZE = 50  # None으로 바꾸면 매칭된 쌍 전체 실행 (오래 걸릴 수 있음)

eval_pairs = pairs if SAMPLE_SIZE is None else random.sample(pairs, min(SAMPLE_SIZE, len(pairs)))
print(f"벤치마크 대상: {len(eval_pairs)}개")

In [ ]:
def infer_faster_whisper(audio_path: str) -> str:
    segments, _ = fw_model.transcribe(audio_path, language="ko", beam_size=5)
    return "".join(seg.text for seg in segments)

def infer_sensevoice(audio_path: str) -> str:
    result = sv_model.generate(
        input=audio_path,
        cache={},
        language="auto",
        use_itn=True,
        batch_size_s=60,
    )
    return rich_transcription_postprocess(result[0]["text"])

In [ ]:
import time
import librosa
from jiwer import cer as jiwer_cer

def run_benchmark(model_name: str, infer_fn, pairs):
    rows = []
    for audio_path, label_path in pairs:
        try:
            reference = load_transcript(label_path)
            duration_sec = librosa.get_duration(path=str(audio_path))

            start = time.perf_counter()
            hypothesis = infer_fn(str(audio_path))
            latency_sec = time.perf_counter() - start

            ref_norm = normalize_text(reference)
            hyp_norm = normalize_text(hypothesis)
            error_rate = jiwer_cer(ref_norm, hyp_norm) if ref_norm else float("nan")

            rows.append({
                "model": model_name,
                "file": audio_path.name,
                "duration_sec": duration_sec,
                "latency_sec": latency_sec,
                "rtf": latency_sec / duration_sec if duration_sec > 0 else float("nan"),
                "cer": error_rate,
                "reference": reference,
                "hypothesis": hypothesis,
            })
        except Exception as e:
            print(f"[{model_name}] {audio_path.name} 처리 실패: {e}")
    return rows

fw_rows = run_benchmark("faster-whisper-small", infer_faster_whisper, eval_pairs)
sv_rows = run_benchmark("SenseVoiceSmall", infer_sensevoice, eval_pairs)

## 5. 결과 집계 및 저장

In [ ]:
import pandas as pd
from tabulate import tabulate

results_df = pd.DataFrame(fw_rows + sv_rows)

summary_df = (
    results_df
    .groupby("model")[["duration_sec", "latency_sec", "rtf", "cer"]]
    .mean()
    .rename(columns={
        "duration_sec": "avg_duration_sec",
        "latency_sec": "avg_latency_sec",
        "rtf": "avg_rtf",
        "cer": "avg_cer",
    })
)

print(tabulate(summary_df, headers="keys", tablefmt="github", floatfmt=".4f"))

results_df.to_csv("stt_benchmark_results.csv", index=False, encoding="utf-8-sig")
print("\nSaved: stt_benchmark_results.csv")